# NeoOLAF native EventStoryLine layer ablation — one document v1.4

This v1.4 notebook keeps the same **OWL-Time** ontology used in RAGTree and the controlled labels `PRECONDITION`/`FALLING_ACTION`. It focuses on the relation bottleneck without modifying `src/neoolaf`:

- parallel sentence event extraction plus **three** whole-document coverage reviews;
- deterministic adaptive **unordered pair coverage** with zero relation-generation LLM calls;
- batched five-way relation decisions that jointly choose existence, direction, and class;
- exact local context, schema-exact positive/negative/reversed examples, evidence requirements, recovery, caching, and `NONE` filtering before Layer 3;
- exact-span-only projected metrics plus strict native span metrics, candidate-pool recall, confusion matrix, and detailed first-failure tracing.

Gold remains unavailable until after Layer 12.


In [1]:
from __future__ import annotations

import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from eventstoryline_native_ablation_v1_4 import (
    RELATION_IDS,
    analyze_run,
    gold_event_index,
    indexed_token_table,
    load_layer_states,
    project_event_label,
    read_json,
    read_jsonl,
    run_native_pipeline,
    seed_ontology_summary,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF


/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_input_v1.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_gold_v1.jsonl"
SMOKE5_INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_input_v1.jsonl"
SMOKE5_GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_gold_v1.jsonl"

# Same seed ontology used by the RAGTree EventStoryLine experiments.
# Override explicitly with EVENTSTORYLINE_ONTOLOGY_PATH when needed.
ONTOLOGY_PATH = first_existing_path(
    "EVENTSTORYLINE_ONTOLOGY_PATH",
    [
        PROJECT_ROOT.parent / "ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT.parent / "RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../RAGTree/data/ontology/OWLTime/time.ttl",
    ],
)

# Controlled normalized benchmark relation schema. This is not the seed ontology.
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/eventstoryline_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/eventstoryline_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_profile_native_ablation_v1_4.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_eventstoryline_native_ablation_v1_4.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_task_guidance_v1_4.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/eventstoryline_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "document_1_10ecbplus_v1_4_owltime_batched_five_way"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Pair batches are independent and run concurrently in Layer 2.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Gold:", GOLD_JSONL)
print("OWL-Time seed ontology:", ONTOLOGY_PATH)
print("Controlled relation catalog:", RELATION_CATALOG)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Task guidance:", TASK_GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


Input: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_input_v1.jsonl
Gold: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_gold_v1.jsonl
OWL-Time seed ontology: /home/galencarmedeiro/git/postdoc/ragtree/data/ontology/OWLTime/time.ttl
Controlled relation catalog: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/ontology/eventstoryline_relation_catalog.json
Profile: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_profile_native_ablation_v1_4.json
Guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/guidance_eventstoryline_native_ablation_v1_4.json
Task guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_task_guidance_v1_4.json
Run dir: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v

## 2. Preflight: anti-leakage, OWL-Time, deterministic pair coverage, five-document files, and event identity


In [3]:
required = [
    INPUT_JSONL, GOLD_JSONL, SMOKE5_INPUT_JSONL, SMOKE5_GOLD_JSONL,
    ONTOLOGY_PATH, RELATION_CATALOG, RELATION_ALIASES,
    PROFILE_PATH, GUIDANCE_PATH, TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
smoke_input_rows = read_jsonl(SMOKE5_INPUT_JSONL)
smoke_gold_rows = read_jsonl(SMOKE5_GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert len(smoke_input_rows) == 5 and len(smoke_gold_rows) == 5
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert all("entities" not in row and "relations" not in row for row in smoke_input_rows)
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]
assert [x["document_id"] for x in smoke_input_rows] == [x["document_id"] for x in smoke_gold_rows]

profile = read_json(PROFILE_PATH)
task = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)
gold = gold_rows[0]
seed_summary = seed_ontology_summary(ONTOLOGY_PATH)
l1_cfg = profile["layers"]["layer01_linguistic_expression_extraction"]
l2_cfg = profile["layers"]["layer02_candidate_enrichment"]

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Sentences:", len(input_rows[0]["sentences"]))
print("Source tokens:", sum(len(x) for x in input_rows[0]["tokens"]))
print("Gold events (not exposed to pipeline):", len(gold["entities"]))
print("Gold evaluated relations:", sum(len(v) for k, v in gold["relations"].items() if k in RELATION_IDS))
print("Ignored null pairs:", len(gold["relations"].get("null", [])))
print("OWL-Time classes loaded:", seed_summary["class_count"])
print("OWL-Time properties loaded:", seed_summary["property_count"])
print("Controlled task relations:", catalog["property_count"])
print("Relation IDs:", task["allowed_relation_ids"])
print("Layer 1 sentence workers:", l1_cfg["sentence_workers"])
print("Layer 1 coverage reviews:", l1_cfg["coverage_review_passes"])
print("Pair strategy:", l1_cfg["pair_strategy"])
print("Exhaustive-event threshold:", l1_cfg["exhaustive_event_threshold"])
print("Relation-generation LLM calls:", l1_cfg["relation_generation_llm_calls"])
print("Layer 2 pair batch size:", l2_cfg["pair_batch_size"])
print("Layer 2 pair batch workers:", l2_cfg["pair_batch_workers"])
print("Five-way decisions:", l2_cfg["five_way_decisions"])
print("Positive evidence required:", l2_cfg["evidence_required_for_positive"])
print("NONE filtered before Layer 3:", l2_cfg["filter_none_before_layer03"])
print("Global-trigger projection fallback:", profile["benchmark_projection"]["fallback_unique_trigger"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))
print("Five-document input IDs:", [row["document_id"] for row in smoke_input_rows])

assert seed_summary["class_count"] > 0 or seed_summary["property_count"] > 0
assert catalog["property_count"] == 2
assert set(task["allowed_relation_ids"]) == set(RELATION_IDS)
assert profile["relations"]["allowed"] == []
assert l1_cfg["relation_generation_llm_calls"] == 0
assert l1_cfg["deterministic_unordered_pair_pool"] is True
assert l2_cfg["filter_none_before_layer03"] is True
assert l2_cfg["evidence_required_for_positive"] is True
assert profile["benchmark_projection"]["fallback_unique_trigger"] is False
assert profile["anti_cheating"]["direct_eventstoryline_extraction"] is False
assert profile["anti_cheating"]["source_event_anchoring"] is False
assert profile["anti_cheating"]["gold_pair_hints"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert profile["benchmark_projection"]["same_seed_ontology_as_ragtree"] is True

# Prompt examples are synthetic and schema-exact; no dataset gold pair is embedded.
assert len(task["five_way_schema_examples"]) >= 4
assert {x["output"]["decision"] for x in task["five_way_schema_examples"]} >= {
    "A_PRECONDITION_B", "A_FALLING_ACTION_B", "B_PRECONDITION_A", "NONE"
}

display(Markdown("### Indexed source table used by Layer 1"))
print(indexed_token_table(input_rows[0]["sentences"], input_rows[0]["tokens"]))

assert profile["anti_cheating"]["gold_event_lexicon"] is False
assert profile["anti_cheating"]["gold_event_count"] is False
assert profile["anti_cheating"]["gold_relation_count"] is False


Document: EventStoryLine - 1_10ecbplus - 1_10ecbplus
Source characters: 745
Sentences: 6
Source tokens: 147
Gold events (not exposed to pipeline): 15
Gold evaluated relations: 20
Ignored null pairs: 1
OWL-Time classes loaded: 23
OWL-Time properties loaded: 62
Controlled task relations: 2
Relation IDs: ['PRECONDITION', 'FALLING_ACTION']
Layer 1 sentence workers: 8
Layer 1 coverage reviews: 3
Pair strategy: adaptive
Exhaustive-event threshold: 25
Relation-generation LLM calls: 0
Layer 2 pair batch size: 16
Layer 2 pair batch workers: 8
Five-way decisions: ['A_PRECONDITION_B', 'A_FALLING_ACTION_B', 'B_PRECONDITION_A', 'B_FALLING_ACTION_A', 'NONE']
Positive evidence required: True
NONE filtered before Layer 3: True
Global-trigger projection fallback: False
Mention-free relation schemas injected: 0
Five-document input IDs: ['EventStoryLine - 1_10ecbplus', 'EventStoryLine - 1_11ecbplus', 'EventStoryLine - 1_12ecbplus', 'EventStoryLine - 1_13ecbplus', 'EventStoryLine - 1_14ecbplus']


### Indexed source table used by Layer 1

[S0] http : / / articles . latimes . com / 2013 / may / 03 / local / la - me - 0504 - lohan - rehab - 20130504
TOKENS 0=http 1=: 2=/ 3=/ 4=articles 5=. 6=latimes 7=. 8=com 9=/ 10=2013 11=/ 12=may 13=/ 14=03 15=/ 16=local 17=/ 18=la 19=- 20=me 21=- 22=0504 23=- 24=lohan 25=- 26=rehab 27=- 28=20130504

[S1] Lindsay Lohan checks into Betty Ford Center
TOKENS 0=Lindsay 1=Lohan 2=checks 3=into 4=Betty 5=Ford 6=Center

[S2] May 03 , 2013
TOKENS 0=May 1=03 2=, 3=2013

[S3] After skipping out on entering a Newport Beach rehabilitation facility and facing the prospect of arrest for violating her probation , Lindsay Lohan has checked into the Betty Ford Center to begin a 90 - day court - mandated stay in her reckless driving conviction .
TOKENS 0=After 1=skipping 2=out 3=on 4=entering 5=a 6=Newport 7=Beach 8=rehabilitation 9=facility 10=and 11=facing 12=the 13=prospect 14=of 15=arrest 16=for 17=violating 18=her 19=probation 20=, 21=Lindsay 22=Lohan 23=has 24=checked 25=into 26=the 27=Betty 28=Fo

## 3. Run the full native Layer 0--12 pipeline

In [4]:
if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        task_guidance_path=TASK_GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)

[NeoOLAF] Run directory: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_4_owltime_batched_five_way
[NeoOLAF] from_layer=0, to_layer=None, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.00s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_lin

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.01s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=8; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.01s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.00s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 15 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 33 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.00s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[Ne

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.00s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=16; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.00s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_4_owltime_batched_five_way/exports
[NeoOLAF] Finished layer: layer12_serialization in 0.04s
[NeoOLAF] Pipeline finished in 126.79s
[NeoOLAF] Saved checkpoint: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_4_owltime_batched_five_way/checkpoints/after_selected_pipeline.pkl.gz
[

### Runtime evidence saved

- `run_manifest.json` records OWL-Time and the v1.4 operational/scientific fingerprint.
- `run_logs/layer01_event_inventory.json` records all accepted/rejected event spans.
- `run_logs/layer01_pair_pool.json` records the deterministic initial unordered pair pool.
- `run_logs/layer02_closure_pair_pool.json` records optional two-hop candidate-closure pairs.
- `run_logs/layer02_relation_decisions.json` records accepted, `NONE`, invalid, reversed and recovered decisions.
- `run_logs/layer02_compact_prompt_audit.json` and `layer02_batch_audit.json` record bounded batch execution.
- `run_logs/layer02_batch_cache/` stores resumable prompt-fingerprint responses.
- Layer 3 receives only validated positive controlled relations; `NONE` never propagates.
- Layer 2/4 decisions, ontology retrieval, API responses, and all Layer 0--12 states remain saved.


## 4. Projected benchmark metrics and strict native span metrics


In [5]:
summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Projected benchmark relation evaluation; exact-span projection, null excluded")
display(pd.DataFrame([summary["strict_relation_evaluation"]]))

print("Strict native span relation evaluation; unmapped native predictions count as false positives")
display(pd.DataFrame([summary["native_span_relation_evaluation"]]))

print("Projected event-ID evaluation")
display(pd.DataFrame([summary["event_entity_evaluation"]]))

print("Strict native event-span evaluation")
display(pd.DataFrame([summary["native_span_event_evaluation"]]))

print("Relation-endpoint projected event evaluation")
display(pd.DataFrame([summary["relation_endpoint_evaluation"]]))

print("Relation candidate-pool coverage")
display(pd.DataFrame([summary["candidate_pool_evaluation"]]))

print("Per-relation projected metrics")
display(pd.DataFrame(summary["per_relation_metrics"]))

print("Direction/class confusion matrix")
display(pd.DataFrame(summary["relation_confusion_matrix"]))

print("Cumulative evaluation")
display(pd.DataFrame(summary["cumulative_evaluation"]))

print("First-failure counts")
print(summary["failure_counts"])


,layer,layer_name,linguistic_expressions,enriched_expressions,entity_candidates,relation_candidates,attribute_candidates,event_candidates,candidate_relation_assertions,candidate_triples,concept_candidates,ontology_relation_candidates,concept_hierarchy_links,relation_hierarchy_links,axiom_schema_candidates,general_axiom_candidates,completion_candidates,validation_issues,reasoning_inferred_triples
0,0,layer00_preprocessing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,layer01_linguistic_expression_extraction,120,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,layer02_candidate_enrichment,120,48,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,layer03_candidate_typing_resolution,120,48,0,33,0,15,0,0,0,0,0,0,0,0,0,0,0
4,4,layer04_candidate_relation_extraction,120,48,0,33,0,15,33,0,0,0,0,0,0,0,0,0,0
5,5,layer05_candidate_triple_generation,120,48,0,33,0,15,33,33,0,0,0,0,0,0,0,0,0
6,6,layer06_concept_relation_induction,120,48,0,33,0,15,33,33,0,2,0,0,0,0,0,0,0
7,7,layer07_hierarchisation,120,48,0,33,0,15,33,33,0,2,0,2,0,0,0,0,0
8,8,layer08_axiom_schemata_extraction,120,48,0,33,0,15,33,33,0,2,0,2,6,0,0,0,0
9,9,layer09_general_axiom_extraction,120,48,0,33,0,15,33,33,0,2,0,2,6,8,0,0,0


Projected benchmark relation evaluation; exact-span projection, null excluded


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,9,20,1,8,19,0.111111,0.05,0.068966


Strict native span relation evaluation; unmapped native predictions count as false positives


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,33,20,1,32,19,0.030303,0.05,0.037736


...

Direction/class confusion matrix


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,1,0,2,0,0,4,0
1,FALLING_ACTION,0,0,3,1,0,9,0


Cumulative evaluation


,layer,layer_name,projected_relation_predicted,projected_relation_gold,projected_relation_true_positive,projected_relation_false_positive,projected_relation_false_negative,projected_relation_precision,projected_relation_recall,projected_relation_f1,...,projected_event_recall,projected_event_f1,native_span_event_predicted,native_span_event_gold,native_span_event_true_positive,native_span_event_false_positive,native_span_event_false_negative,native_span_event_precision,native_span_event_recall,native_span_event_f1
0,0,layer00_preprocessing,0,20,0,0,20,0.000000,0.00,0.000000,...,0.0,0.00,0,15,0,0,15,0.0,0.0,0.0
1,1,layer01_linguistic_expression_extraction,0,20,0,0,20,0.000000,0.00,0.000000,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
2,2,layer02_candidate_enrichment,0,20,0,0,20,0.000000,0.00,0.000000,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
3,3,layer03_candidate_typing_resolution,0,20,0,0,20,0.000000,0.00,0.000000,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
4,4,layer04_candidate_relation_extraction,9,20,1,8,19,0.111111,0.05,0.068966,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
5,5,layer05_candidate_triple_generation,9,20,1,8,19,0.111111,0.05,0.068966,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
6,6,layer06_concept_relation_induction,9,20,1,8,19,0.111111,0.05,0.068966,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
7,7,layer07_hierarchisation,9,20,1,8,19,0.111111,0.05,0.068966,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
8,8,layer08_axiom_schemata_extraction,9,20,1,8,19,0.111111,0.05,0.068966,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6
9,9,layer09_general_axiom_extraction,9,20,1,8,19,0.111111,0.05,0.068966,...,0.6,0.75,15,15,9,6,6,0.6,0.6,0.6


First-failure counts
{'target_event_missing': 7, 'classified_none': 5, 'source_event_missing': 6, 'survived_to_layer05': 1, 'wrong_direction': 1}


## 5. Layer 1 validated event inventory and deterministic unordered relation-pair pool


In [6]:
layer1_rows = read_json(RUN_DIR / "run_logs/layer01_event_relation_instances.json")
layer1_df = pd.DataFrame(layer1_rows)
display(layer1_df)

accepted = layer1_df[layer1_df["status"] == "accepted"] if not layer1_df.empty else layer1_df
if not accepted.empty:
    print("Accepted event mentions:", int((accepted["label"] == "event_mention").sum()))
    print("Deterministic pair expressions:", int((accepted["label"] == "relation_instance").sum()))
    print("Rejected rows:", int((layer1_df["status"] == "rejected").sum()))


,phase,status,reason,text,label,repair_steps,expr_id,justification,pair_id,candidate_reasons,strategy
0,sentence_inventory_4,rejected,connective_or_auxiliary_only,was,event_mention,[],NaN,NaN,NaN,NaN,NaN
1,materialization,accepted,NaN,S1[2:4]::checks into,event_mention,NaN,expr_00000,Explicit phrasal verb indicating an action.,NaN,NaN,NaN
2,materialization,accepted,NaN,S3[1:3]::skipping out,event_mention,NaN,expr_00001,Explicit phrasal verb.,NaN,NaN,NaN
3,materialization,accepted,NaN,S3[4:5]::entering,event_mention,NaN,expr_00002,Explicit verb.,NaN,NaN,NaN
4,materialization,accepted,NaN,S3[11:12]::facing,event_mention,NaN,expr_00003,Explicit verb.,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
116,pair_pool_materialization,accepted,NaN,S5[9:12]::rear - ended || potentially related ...,relation_instance,NaN,expr_00115,NaN,P00100,[exhaustive_unordered_pair],exhaustive
117,pair_pool_materialization,accepted,NaN,S5[9:12]::rear - ended || potentially related ...,relation_instance,NaN,expr_00116,NaN,P00101,[exhaustive_unordered_pair],exhaustive
118,pair_pool_materialization,accepted,NaN,S5[27:28]::lied || potentially related || S5[3...,relation_instance,NaN,expr_00117,NaN,P00102,[exhaustive_unordered_pair],exhaustive
119,pair_pool_materialization,accepted,NaN,S5[27:28]::lied || potentially related || S5[3...,relation_instance,NaN,expr_00118,NaN,P00103,[exhaustive_unordered_pair],exhaustive


Accepted event mentions: 15
Deterministic pair expressions: 105
Rejected rows: 1


### Layer 1A span validation/repair and Layer 1 deterministic pair audit


In [7]:
inventory_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_event_inventory.json"))
pair_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_pair_pool.json"))
call_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_call_audit.json"))

print("Layer 1A event inventory audit")
display(inventory_audit)
if not inventory_audit.empty:
    print("Accepted validated proposals:", int((inventory_audit["status"] == "accepted").sum()))
    print("Rejected event proposals:", int((inventory_audit["status"] == "rejected").sum()))
    if "repaired" in inventory_audit:
        print("Source-token span repairs:", int(inventory_audit["repaired"].fillna(False).sum()))

print("Deterministic initial unordered pair pool")
display(pair_pool)
if not pair_pool.empty:
    print("Initial pair candidates:", len(pair_pool))
    print("Pair strategy values:", sorted(pair_pool["strategy"].dropna().unique()))

print("Layer 1 sentence/review/pair construction audit")
display(call_audit)
if not call_audit.empty:
    display(call_audit.groupby(["phase", "status"], dropna=False).size().reset_index(name="calls"))


Layer 1A event inventory audit


,raw_item,proposed_event_key,proposed_sentence_id,proposed_token_start,proposed_token_end,proposed_trigger,repair_steps,status,reason,canonical_event_key,sentence_id,token_start,token_end,repaired_trigger,repaired,phase,justification,repaired_tokens
0,"{'sentence_id': 1, 'token_start': 2, 'token_en...",None,1,2,4,checks into,[],accepted,validated_source_token_span,S1[2:4]::checks into,1.0,2.0,4.0,checks into,False,sentence_inventory_1,Explicit phrasal verb indicating an action.,NaN
1,"{'sentence_id': 3, 'token_start': 1, 'token_en...",None,3,1,3,skipping out,[],accepted,validated_source_token_span,S3[1:3]::skipping out,3.0,1.0,3.0,skipping out,False,sentence_inventory_3,Explicit phrasal verb.,NaN
2,"{'sentence_id': 3, 'token_start': 4, 'token_en...",None,3,4,5,entering,[],accepted,validated_source_token_span,S3[4:5]::entering,3.0,4.0,5.0,entering,False,sentence_inventory_3,Explicit verb.,NaN
3,"{'sentence_id': 3, 'token_start': 11, 'token_e...",None,3,11,12,facing,[],accepted,validated_source_token_span,S3[11:12]::facing,3.0,11.0,12.0,facing,False,sentence_inventory_3,Explicit verb.,NaN
4,"{'sentence_id': 3, 'token_start': 17, 'token_e...",None,3,17,18,violating,[],accepted,validated_source_token_span,S3[17:18]::violating,3.0,17.0,18.0,violating,False,sentence_inventory_3,Explicit verb.,NaN
5,"{'sentence_id': 3, 'token_start': 24, 'token_e...",None,3,24,25,checked,[completed_phrasal_verb_particle],accepted,validated_source_token_span,S3[24:26]::checked into,3.0,24.0,26.0,checked into,True,sentence_inventory_3,Explicit verb.,NaN
6,"{'sentence_id': 3, 'token_start': 31, 'token_e...",None,3,31,32,begin,[],accepted,validated_source_token_span,S3[31:32]::begin,3.0,31.0,32.0,begin,False,sentence_inventory_3,Explicit verb.,NaN
7,"{'sentence_id': 4, 'token_start': 1, 'token_en...",None,4,1,2,was,[],rejected,connective_or_auxiliary_only,NaN,NaN,NaN,NaN,NaN,NaN,sentence_inventory_4,Explicit state of being.,[was]
8,"{'sentence_id': 4, 'token_start': 15, 'token_e...",None,4,15,16,violation,[],accepted,validated_source_token_span,S4[15:16]::violation,4.0,15.0,16.0,violation,False,sentence_inventory_4,Explicit eventive noun.,NaN
9,"{'sentence_id': 4, 'token_start': 19, 'token_e...",None,4,19,22,drunk driving case,[],accepted,validated_source_token_span,S4[19:22]::drunk driving case,4.0,19.0,22.0,drunk driving case,False,sentence_inventory_4,Explicit eventive noun phrase.,NaN


Accepted validated proposals: 17
Rejected event proposals: 1
Source-token span repairs: 2
Deterministic initial unordered pair pool


,event_a_id,event_b_id,event_a_key,event_b_key,event_a_sentence,event_b_sentence,candidate_reasons,strategy,phase,pair_id
0,E0000,E0001,S1[2:4]::checks into,S3[1:3]::skipping out,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00000
1,E0000,E0002,S1[2:4]::checks into,S3[4:5]::entering,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00001
2,E0000,E0003,S1[2:4]::checks into,S3[11:12]::facing,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00002
3,E0000,E0004,S1[2:4]::checks into,S3[17:18]::violating,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00003
4,E0000,E0005,S1[2:4]::checks into,S3[24:26]::checked into,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00004
...,...,...,...,...,...,...,...,...,...,...
100,E0011,E0013,S5[9:12]::rear - ended,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00100
101,E0011,E0014,S5[9:12]::rear - ended,S5[34:37]::was not driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00101
102,E0012,E0013,S5[27:28]::lied,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00102
103,E0012,E0014,S5[27:28]::lied,S5[34:37]::was not driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00103


Initial pair candidates: 105
Pair strategy values: ['exhaustive']
Layer 1 sentence/review/pair construction audit


,phase,sentence_id,status,reason,proposals,new_validated_events,pass_index,inventory_size_after,strategy,event_count,pair_count,exhaustive_event_threshold
0,sentence_inventory,0.0,skipped,url_sentence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sentence_inventory,2.0,skipped,date_or_publication_metadata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sentence_inventory,1.0,ok,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,sentence_inventory,3.0,ok,NaN,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
4,sentence_inventory,4.0,ok,NaN,3.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
5,sentence_inventory,5.0,ok,NaN,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN
6,coverage_review,NaN,ok,NaN,1.0,0.0,1.0,14.0,NaN,NaN,NaN,NaN
7,coverage_review,NaN,ok,NaN,2.0,1.0,2.0,15.0,NaN,NaN,NaN,NaN
8,coverage_review,NaN,ok,NaN,0.0,0.0,3.0,15.0,NaN,NaN,NaN,NaN
9,deterministic_pair_pool,NaN,ok,NaN,NaN,NaN,NaN,NaN,exhaustive,15.0,105.0,25.0


,phase,status,calls
0,coverage_review,ok,3
1,deterministic_pair_pool,ok,1
2,sentence_inventory,ok,4
3,sentence_inventory,skipped,2


## 6. Batched five-way relation decisions, recovery, closure and filtering


In [8]:
layer2 = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_decisions.json"))
prompt_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))
batch_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_batch_audit.json"))
closure_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_closure_pair_pool.json"))

display(layer2)
if not layer2.empty:
    print("Layer 2 decision status counts")
    display(layer2.groupby(["status", "decision"], dropna=False).size().reset_index(name="pairs"))
    accepted_l2 = layer2[layer2["status"] == "accepted"]
    print("Positive relations entering Layer 3:", len(accepted_l2))
    print("Filtered NONE/invalid relations:", len(layer2) - len(accepted_l2))

print("Compact batch prompt audit")
display(prompt_audit)
if not prompt_audit.empty:
    print("Prompted pair decisions:", int(prompt_audit["pair_count"].sum()))
    print("Mean pairs per prompt:", round(prompt_audit["pair_count"].mean(), 2))
    print("Mean Layer 2 user characters:", round(prompt_audit["user_chars"].mean(), 1))
    print("Maximum Layer 2 user characters:", int(prompt_audit["user_chars"].max()))

print("Batch/recovery/cache audit")
display(batch_audit)
print("Two-hop closure candidate pool")
display(closure_pool)


,pair_id,phase,event_a,event_b,decision,found,status,evidence_sentence_ids,evidence_text,reason,confidence,expr_id,source,target,selected_relation_id,ontology_hints,recovery
0,P00000,initial,S1[2:4]::checks into,S3[1:3]::skipping out,NONE,False,filtered_none,[],,No causal or narrative link between checking i...,0.90,NaN,NaN,NaN,NaN,NaN,NaN
1,P00001,initial,S1[2:4]::checks into,S3[4:5]::entering,NONE,False,filtered_none,[],,No causal or narrative link between checking i...,0.90,NaN,NaN,NaN,NaN,NaN,NaN
2,P00002,initial,S1[2:4]::checks into,S3[11:12]::facing,NONE,False,filtered_none,[],,No causal or narrative link between checking i...,0.90,NaN,NaN,NaN,NaN,NaN,NaN
3,P00003,initial,S1[2:4]::checks into,S3[17:18]::violating,NONE,False,filtered_none,[],,No causal or narrative link between checking i...,0.90,NaN,NaN,NaN,NaN,NaN,NaN
4,P00004,initial,S1[2:4]::checks into,S3[24:26]::checked into,NONE,False,filtered_none,[],,No causal or narrative link between checking i...,0.90,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,P00100,initial,S5[9:12]::rear - ended,S5[31:32]::telling,NONE,False,filtered_none,[],,No direct link between the rear-ended event an...,0.85,NaN,NaN,NaN,NaN,NaN,NaN
105,P00101,initial,S5[9:12]::rear - ended,S5[34:37]::was not driving,NONE,False,filtered_none,[],,No direct link between the rear-ended event an...,0.85,NaN,NaN,NaN,NaN,NaN,NaN
106,P00102,initial,S5[27:28]::lied,S5[31:32]::telling,A_FALLING_ACTION_B,True,accepted,[5],Her latest stay at Betty Ford comes after Loha...,"The event ""lied"" is followed by the event ""tel...",0.92,expr_00117,S5[27:28]::lied,S5[31:32]::telling,FALLING_ACTION,"[controlled_relation:FALLING_ACTION, promote_t...",NaN
107,P00103,initial,S5[27:28]::lied,S5[34:37]::was not driving,A_FALLING_ACTION_B,True,accepted,[5],Her latest stay at Betty Ford comes after Loha...,"The event ""lied"" is followed by the event ""was...",0.92,expr_00118,S5[27:28]::lied,S5[34:37]::was not driving,FALLING_ACTION,"[controlled_relation:FALLING_ACTION, promote_t...",NaN


Layer 2 decision status counts


,status,decision,pairs
0,accepted,A_FALLING_ACTION_B,12
1,accepted,A_PRECONDITION_B,13
2,accepted,B_PRECONDITION_A,8
3,filtered_none,NONE,72
4,missing_decision,NaN,4


Positive relations entering Layer 3: 33
Filtered NONE/invalid relations: 76
Compact batch prompt audit


,batch_index,phase,recovery,pair_ids,pair_count,candidate_decisions,system_chars,user_chars
0,0,initial,False,"[P00000, P00001, P00002, P00003, P00004, P0000...",16,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2644,14914
1,1,initial,False,"[P00016, P00017, P00018, P00019, P00020, P0002...",16,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2644,13892
2,2,initial,False,"[P00032, P00033, P00034, P00035, P00036, P0003...",16,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2644,13902
3,3,initial,False,"[P00048, P00049, P00050, P00051, P00052, P0005...",16,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2644,13951
4,4,initial,False,"[P00064, P00065, P00066, P00067, P00068, P0006...",16,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2644,13944
5,5,initial,False,"[P00080, P00081, P00082, P00083, P00084, P0008...",16,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2644,13892
6,6,initial,False,"[P00096, P00097, P00098, P00099, P00100, P0010...",9,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2644,9709
7,7,initial,True,"[P00028, P00029, P00030, P00031]",4,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",2688,8100


Prompted pair decisions: 109
Mean pairs per prompt: 13.62
Mean Layer 2 user characters: 12788.0
Maximum Layer 2 user characters: 14914
Batch/recovery/cache audit


,phase,batch_index,recovery,pair_count,accepted_decisions,unresolved_decisions,status,attempt,cache_path
0,initial,2,False,16,16,0,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
1,initial,4,False,16,16,0,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
2,initial,0,False,16,16,0,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
3,initial,5,False,16,16,0,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
4,initial,6,False,9,9,0,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
5,initial,3,False,16,16,0,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
6,initial,1,False,16,12,4,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
7,initial,7,True,4,4,0,ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...


Two-hop closure candidate pool


""


## 7. Exact gold-relation failure trace and confusion matrix


In [9]:
trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace.csv")
confusion_df = pd.read_csv(RUN_DIR / "analysis/relation_confusion_matrix.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)
display(confusion_df)


,source_key,relation_id,target_key,first_failure,pair_in_pool,decision_status,predicted_decision,predicted_source,predicted_relation,predicted_target
0,S1[2:4]::checks into,FALLING_ACTION,S3[15:16]::arrest,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
1,S1[2:4]::checks into,FALLING_ACTION,S3[4:5]::entering,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
2,S1[2:4]::checks into,PRECONDITION,S3[39:40]::stay,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
3,S1[2:4]::checks into,PRECONDITION,S5[2:3]::stay,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
4,S3[15:16]::arrest,FALLING_ACTION,S3[17:18]::violating,source_event_missing,False,NaN,NaN,NaN,NaN,NaN
5,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into,source_event_missing,False,NaN,NaN,NaN,NaN,NaN
6,S3[24:26]::checked into,PRECONDITION,S3[39:40]::stay,survived_to_layer05,True,accepted,A_PRECONDITION_B,S3[24:26]::checked into,PRECONDITION,S3[39:40]::stay
7,S3[24:26]::checked into,PRECONDITION,S5[2:3]::stay,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
8,S3[39:40]::stay,FALLING_ACTION,S3[44:45]::conviction,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
9,S3[39:40]::stay,FALLING_ACTION,S5[27:28]::lied,classified_none,True,filtered_none,NONE,NaN,NaN,NaN


,first_failure,gold_relations
3,target_event_missing,7
1,source_event_missing,6
0,classified_none,5
2,survived_to_layer05,1
4,wrong_direction,1


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,1,0,2,0,0,4,0
1,FALLING_ACTION,0,0,3,1,0,9,0


## 8. Native event candidates, assertions and triples

In [10]:
states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    print("Layer 3 event candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "ontology_hints": c.ontology_hints,
    } for c in layer3.event_candidates or []]))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)

Layer 3 event candidates


,candidate_id,canonical_label,mentions,ontology_hints
0,cand_s_00000,S1[2:4]::checks into,[S1[2:4]::checks into],"[semantic_role:event_mention, candidate_family..."
1,cand_s_00001,S3[1:3]::skipping out,[S3[1:3]::skipping out],"[semantic_role:event_mention, candidate_family..."
2,cand_s_00002,S3[4:5]::entering,[S3[4:5]::entering],"[semantic_role:event_mention, candidate_family..."
3,cand_s_00003,S3[11:12]::facing,[S3[11:12]::facing],"[semantic_role:event_mention, candidate_family..."
4,cand_s_00004,S3[17:18]::violating,[S3[17:18]::violating],"[semantic_role:event_mention, candidate_family..."
5,cand_s_00005,S3[24:26]::checked into,[S3[24:26]::checked into],"[semantic_role:event_mention, candidate_family..."
6,cand_s_00006,S3[31:32]::begin,[S3[31:32]::begin],"[semantic_role:event_mention, candidate_family..."
7,cand_s_00007,S3[39:40]::stay,[S3[39:40]::stay],"[semantic_role:event_mention, candidate_family..."
8,cand_s_00008,S4[15:16]::violation,[S4[15:16]::violation],"[semantic_role:event_mention, candidate_family..."
9,cand_s_00009,S4[19:22]::drunk driving case,[S4[19:22]::drunk driving case],"[semantic_role:event_mention, candidate_family..."


Layer 3 relation candidates


,candidate_id,canonical_label,mentions,controlled_hints
0,cand_r_00000,PRECONDITION,[S3[1:3]::skipping out || PRECONDITION || S3[1...,[controlled_relation:PRECONDITION]
1,cand_r_00001,PRECONDITION,[S3[1:3]::skipping out || PRECONDITION || S3[2...,[controlled_relation:PRECONDITION]
2,cand_r_00002,PRECONDITION,[S3[1:3]::skipping out || PRECONDITION || S3[3...,[controlled_relation:PRECONDITION]
3,cand_r_00003,PRECONDITION,[S3[1:3]::skipping out || PRECONDITION || S3[3...,[controlled_relation:PRECONDITION]
4,cand_r_00004,PRECONDITION,[S3[1:3]::skipping out || PRECONDITION || S4[1...,[controlled_relation:PRECONDITION]
5,cand_r_00005,PRECONDITION,[S3[4:5]::entering || PRECONDITION || S3[11:12...,[controlled_relation:PRECONDITION]
6,cand_r_00006,PRECONDITION,[S3[17:18]::violating || PRECONDITION || S3[24...,[controlled_relation:PRECONDITION]
7,cand_r_00007,PRECONDITION,[S3[17:18]::violating || PRECONDITION || S3[31...,[controlled_relation:PRECONDITION]
8,cand_r_00008,PRECONDITION,[S3[17:18]::violating || PRECONDITION || S3[39...,[controlled_relation:PRECONDITION]
9,cand_r_00009,PRECONDITION,[S3[24:26]::checked into || PRECONDITION || S3...,[controlled_relation:PRECONDITION]


Layer 4 assertions


,source,predicate,target,confidence
0,S3[1:3]::skipping out,PRECONDITION,S3[17:18]::violating,1.0
1,S3[1:3]::skipping out,PRECONDITION,S3[24:26]::checked into,1.0
2,S3[1:3]::skipping out,PRECONDITION,S3[31:32]::begin,1.0
3,S3[1:3]::skipping out,PRECONDITION,S3[39:40]::stay,1.0
4,S3[1:3]::skipping out,PRECONDITION,S4[15:16]::violation,1.0
5,S3[4:5]::entering,PRECONDITION,S3[11:12]::facing,1.0
6,S3[17:18]::violating,PRECONDITION,S3[24:26]::checked into,1.0
7,S3[17:18]::violating,PRECONDITION,S3[31:32]::begin,1.0
8,S3[17:18]::violating,PRECONDITION,S3[39:40]::stay,1.0
9,S3[24:26]::checked into,PRECONDITION,S3[31:32]::begin,1.0


Layer 5 triples


,subject,predicate,object,confidence
0,S3[1:3]::skipping out,PRECONDITION,S3[17:18]::violating,1.0
1,S3[1:3]::skipping out,PRECONDITION,S3[24:26]::checked into,1.0
2,S3[1:3]::skipping out,PRECONDITION,S3[31:32]::begin,1.0
3,S3[1:3]::skipping out,PRECONDITION,S3[39:40]::stay,1.0
4,S3[1:3]::skipping out,PRECONDITION,S4[15:16]::violation,1.0
5,S3[4:5]::entering,PRECONDITION,S3[11:12]::facing,1.0
6,S3[17:18]::violating,PRECONDITION,S3[24:26]::checked into,1.0
7,S3[17:18]::violating,PRECONDITION,S3[31:32]::begin,1.0
8,S3[17:18]::violating,PRECONDITION,S3[39:40]::stay,1.0
9,S3[24:26]::checked into,PRECONDITION,S3[31:32]::begin,1.0


Layer 6 ontology relation candidates: 2
Layer 11 completion candidates: 0


## 9. Exact-span-only event projection audit


In [11]:
projection = pd.read_csv(RUN_DIR / "analysis/event_projection_audit.csv")
display(projection)

# Exact-span demonstration using source structure. Gold is consulted only here,
# after the pipeline has completed. Trigger-only fallbacks are disabled.
first_gold_key = next(iter(gold_event_index(gold)["keys_by_id"].values()))[0]
print(first_gold_key)
print(project_event_label(first_gold_key, gold))


,event_id,method,label,candidate_event_ids
0,EVENT_34cccabacad4dea93d3e762114dc05cd,exact_sentence_token_span,S1[2:4]::checks into,NaN
1,NaN,unmapped_or_ambiguous_exact_span,S3[1:3]::skipping out,[]
2,EVENT_b79df11f8737f700691af4f8a7132190,exact_sentence_token_span,S3[4:5]::entering,NaN
3,NaN,unmapped_or_ambiguous_exact_span,S3[11:12]::facing,[]
4,EVENT_9c07ef70f303f23dd82b32fb28a61ecb,exact_sentence_token_span,S3[17:18]::violating,NaN
5,EVENT_d7aec1e4ff20c3264f9fb7686355eb8a,exact_sentence_token_span,S3[24:26]::checked into,NaN
6,NaN,unmapped_or_ambiguous_exact_span,S3[31:32]::begin,[]
7,EVENT_2ea9a901cc230afcc371bcffee02a551,exact_sentence_token_span,S3[39:40]::stay,NaN
8,EVENT_434faad0c9ec71baa6a91df3c385299b,exact_sentence_token_span,S4[15:16]::violation,NaN
9,NaN,unmapped_or_ambiguous_exact_span,S4[19:22]::drunk driving case,[]


S4[15:16]::violation
{'event_id': 'EVENT_434faad0c9ec71baa6a91df3c385299b', 'method': 'exact_sentence_token_span', 'label': 'S4[15:16]::violation'}


## 10. Speed, concurrency and errors

In [12]:
def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))

,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,response_chars,response_path,json_parse_ok,parsed_type,parse_error
0,1,layer01_event_inventory_v1_4,openai/gpt-oss-20b,0.0,2,1264,2441,8192,180,2026-08-01 14:33:35,ok,1.202,227,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
1,4,layer01_event_inventory_v1_4,openai/gpt-oss-20b,0.0,2,1264,2811,8192,180,2026-08-01 14:33:35,ok,3.477,823,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
2,3,layer01_event_inventory_v1_4,openai/gpt-oss-20b,0.0,2,1264,2660,8192,180,2026-08-01 14:33:35,ok,4.633,494,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
3,2,layer01_event_inventory_v1_4,openai/gpt-oss-20b,0.0,2,1264,3001,8192,180,2026-08-01 14:33:35,ok,6.370,835,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
4,5,layer01_event_inventory_v1_4,openai/gpt-oss-20b,0.0,2,1265,8886,8192,180,2026-08-01 14:33:41,ok,17.787,228,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
5,6,layer01_event_inventory_v1_4,openai/gpt-oss-20b,0.0,2,1306,8886,8192,180,2026-08-01 14:33:59,ok,2.635,393,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
6,7,layer01_event_inventory_v1_4,openai/gpt-oss-20b,0.0,2,1325,9216,8192,180,2026-08-01 14:34:01,ok,18.641,72,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
7,10,layer02_batched_five_way_relations_v1_4,openai/gpt-oss-20b,0.0,2,2644,13902,4096,180,2026-08-01 14:34:20,ok,3.429,2683,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
8,11,layer02_batched_five_way_relations_v1_4,openai/gpt-oss-20b,0.0,2,2644,13944,4096,180,2026-08-01 14:34:20,ok,7.829,3355,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
9,8,layer02_batched_five_way_relations_v1_4,openai/gpt-oss-20b,0.0,2,2644,14914,4096,180,2026-08-01 14:34:20,ok,9.583,2827,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None


,layer_tag,calls,total_recorded_seconds,maximum_call_seconds,mean_system_chars,mean_user_chars,mean_response_chars
0,layer01_event_inventory_v1_4,7,54.745,18.641,1278.857143,5414.428571,438.857143
1,layer02_batched_five_way_relations_v1_4,8,183.472,75.378,2649.500000,12788.000000,2629.125000


Backend/API errors: 0
JSON parse errors: 0


,layer_name,retrieval_calls
0,layer02_candidate_enrichment,8


Wall-clock pipeline seconds: 126.80513167381287


## Success checklist before the five-document batch

1. The sibling RAGTree OWL-Time file resolves and remains secondary evidence.
2. Three coverage reviews materially improve exact event-span recall, including repeated mentions and eventive nouns.
3. Layer 1 deterministically builds the full unordered pair pool for this small document with **zero** relation-generation LLM calls.
4. Layer 2 classifies bounded pair batches with exactly five decisions, both directions considered, textual evidence required, one recovery pass, and cached responses.
5. `NONE`, conflicting, malformed, and evidence-free outputs are absent from Layer 3 relation candidates.
6. The candidate-pool recall given available endpoints is high; missed relations are separated into event, pool, `NONE`, direction, class, and Layer 4/5 failures.
7. Exact-span-only projected metrics and strict native span metrics are both reported; no global trigger fallback remains.
8. Gold remains unavailable until post-Layer-12 evaluation. Freeze v1.4 only after this one-document result is satisfactory.
